In [ ]:
import pandas as pd
import geopandas as gpd

### administrative pools

In [ ]:

### original pools from the administrative  website
admin_pools = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/baederleben_swimmingpools_coords.csv')
admin_pools['address'] = admin_pools['street'] + ', ' + admin_pools['postal_code'].astype(str) + ' ' + admin_pools['city'] + ', Germany'
admin_pools =admin_pools.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
### remove natural swimming pools
admin_pools = admin_pools.drop(admin_pools[admin_pools['pool_type'] == 'Natürliche Badestelle'].index).reset_index(drop=True)
#admin_pools = admin_pools.drop_duplicates(subset=['address']).reset_index(drop=True)    

admin_pools.info()

In [ ]:
admin_pools.pool_type.value_counts()

In [ ]:
#see duplicated addresses in admin pools    
admin_pools[admin_pools.duplicated(subset=['address'], keep=False)].sort_values(by=['address'])[['pool_name', 'address', 'latitude', 'longitude']]

In [ ]:
pids_geo_latlong = pd.read_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/reverse_geocoding_pool_pids.csv')
pids_geo_latlong.info()

In [ ]:
pids_geo_latlong.to_csv('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/reverse_geocoding_pool_pids.csv', index=False)

In [ ]:
pids_geo_latlong = pd.merge(pids_geo_latlong, admin_pools, on = ['latitude', 'longitude'], how = 'inner')[['place_id', 'latitude', 'longitude']]
pids_geo_latlong.info()

### google collected pids

In [ ]:
#reverse geocode pool locations to get spatial data
pids_geocoded = gpd.read_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/pois_within_shapefile.gpkg')
pids_geocoded =  pids_geocoded[pids_geocoded['place_id'].isin(pids_geo_latlong['place_id'])].reset_index(drop=True)
pids_geocoded.info()

In [ ]:
pids_geocoded.to_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/pois_within_shapefile.gpkg', driver='GPKG')

In [ ]:
pids_geo_latlong[pids_geo_latlong['place_id'].isin(pids_geocoded['place_id'])]

In [ ]:
pids_geocoded[pids_geocoded['place_id'] =='ChIJG2xtGaszmEcR47UnFpMh4h4' ][['place_id', 'unique_identifier', 'place_name', 'latitude', 'longitude', 'address']]

there is a discrepancy in the  coordinates from the admin website and those from google. Using the already collected data points from the geocode reversing we can calculate the distance  between varying coordinates for the same place and then have an estimate of a radius for the match

In [ ]:
## pids initially collected for all of germany
pids_initial = gpd.read_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/clean_datasets/pois_within_shapefile.gpkg')

pids_initial.info()

In [ ]:
#pids collectes with state capital data
pids_gsc = pd.read_csv('/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/relevant_gsc_pools.csv')
pids_gsc.info()

In [ ]:
##merge the google datasets
merged_google_pids = pd.merge(pids_initial, pids_geocoded, on = pids_geocoded.columns.to_list(), how = 'outer')
merged_google_pids = pd.merge(merged_google_pids, pids_gsc, on = ['place_id', 'latitude', 'longitude', 'place_name', 'primary_place_type', 'place_subtype', 'google_url','total_reviews' ], how = 'outer')
merged_google_pids = merged_google_pids.drop_duplicates(subset=['place_id']).reset_index(drop=True)
merged_google_pids = merged_google_pids.drop_duplicates(subset=['latitude', 'longitude']).reset_index(drop=True)
#merged_google_pids = merged_google_pids.dropna(subset=['total_reviews']).reset_index(drop=True)
merged_google_pids = merged_google_pids.dropna(subset=['latitude', 'longitude']).reset_index(drop=True)
merged_google_pids.info()

In [ ]:
merged_google_pids.to_file('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/GERMAN_PUBLIC_POOLS/clean_datasets/merged_google_pids.gpkg', driver='GPKG')

In [ ]:
# see rows with same address but different place_id
merged_google_pids[merged_google_pids.duplicated(subset=['address'], keep=False)].sort_values(by=['address'])[['place_id', 'place_name', 'address', 'primary_place_type', 'place_subtype', 'total_reviews']]    

### Known matches

In [ ]:
# Admin pools
pid_matches = pd.merge(pids_geo_latlong[['place_id', 'latitude', 'longitude']],merged_google_pids[['place_id', 'latitude', 'longitude']], on='place_id', suffixes=('_admin', '_google'))
address_matches = pd.merge(merged_google_pids[['place_id', 'latitude', 'longitude', 'address']], admin_pools[['address', 'latitude', 'longitude']], on='address', suffixes=('_google', '_admin'))
known_matches = pd.concat([pid_matches, address_matches], ignore_index=True).drop_duplicates(subset=['place_id']).reset_index(drop=True)
known_matches = known_matches.drop_duplicates(subset=['latitude_google', 'longitude_google']).reset_index(drop=True)
known_matches = known_matches.drop_duplicates(subset=['latitude_admin', 'longitude_admin']).reset_index(drop=True)
known_matches.info()

### Remove matches from the google data and the admin data

In [ ]:
## pids with certainty . using the know matches and the address field
certain_pids = merged_google_pids[merged_google_pids['place_id'].isin(known_matches['place_id'])]
certain_pids = certain_pids.drop_duplicates(subset=['place_id']).reset_index(drop=True)
certain_pids.info()

In [ ]:
uncertain_pids = merged_google_pids[~merged_google_pids['place_id'].isin(certain_pids['place_id'])]
uncertain_pids.info()   

### admin pools unaccounted for

In [ ]:
def find_true_unique_rows(original_df, pool_df):
    """
    Find rows in original that are not in pool_pids, accounting for coordinate duplicates
    """
    print("=== ANALYZING COORDINATE DUPLICATES ===")
    
    # Count unique coordinates in each dataset
    original_unique_coords = original_df[['latitude', 'longitude']].drop_duplicates()
     
    #change column names to latitude and longitude
    pool_df = pool_df.rename(columns={'latitude_admin':'latitude', 'longitude_admin':'longitude'})
    pool_unique_coords = pool_df[['latitude', 'longitude']].drop_duplicates()

    print(f"Original total rows: {len(original_df)}")
    print(f"Original unique coordinates: {len(original_unique_coords)}")
    print(f"Pool_pids total rows: {len(pool_df)}")
    print(f"Pool_pids unique coordinates: {len(pool_unique_coords)}")
   
    # Find coordinates in original but not in pool
    merged = original_unique_coords.merge(
        pool_unique_coords, 
        on=['latitude', 'longitude'], 
        how='left', 
        indicator=True
    )
    
    unique_coords = merged[merged['_merge'] == 'left_only']
    
    print(f"\nUnique coordinates in original (not in pool): {len(unique_coords)}")
    
    # Get the full rows from original that have these unique coordinates
    # We'll take just the first occurrence of each unique coordinate
    unique_rows = original_df.merge(
        unique_coords[['latitude', 'longitude']], 
        on=['latitude', 'longitude']
    ).drop_duplicates(subset=['latitude', 'longitude'])
    
    return unique_rows

admin_unmatched = find_true_unique_rows(admin_pools, known_matches)
print(f"\nTrue unique rows (first occurrence of each unique coordinate):")
print(admin_unmatched)

### some functions

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import geopy.distance
import matplotlib.pyplot as plt

In [ ]:
def check_coordinate_systems(website_df, google_df):
    """
    Check if coordinates are in expected WGS84 ranges
    """
    print("=== COORDINATE SYSTEM ANALYSIS ===")
    
    # Check latitude ranges (should be -90 to 90)
    print("\n--- Latitude Ranges ---")
    print(f"Website: {website_df['latitude'].min():.6f} to {website_df['latitude'].max():.6f}")
    print(f"Google:  {google_df['latitude'].min():.6f} to {google_df['latitude'].max():.6f}")
    
    # Check longitude ranges (should be -180 to 180)
    print("\n--- Longitude Ranges ---")
    print(f"Website: {website_df['longitude'].min():.6f} to {website_df['longitude'].max():.6f}")
    print(f"Google:  {google_df['longitude'].min():.6f} to {google_df['longitude'].max():.6f}")
    
    # Check for obvious coordinate system issues
    web_lat_outliers = website_df[(website_df['latitude'] < -90) | (website_df['latitude'] > 90)]
    web_lon_outliers = website_df[(website_df['longitude'] < -180) | (website_df['longitude'] > 180)]
    
    print(f"\n--- Potential Coordinate System Issues ---")
    print(f"Website lat outside WGS84 bounds: {len(web_lat_outliers)}")
    print(f"Website lon outside WGS84 bounds: {len(web_lon_outliers)}")
    
    return web_lat_outliers, web_lon_outliers

In [ ]:
def check_systematic_offsets(known_matches_df):
    """
    Check if there's a consistent offset pattern that suggests coordinate system mismatch
    """
    print("\n=== SYSTEMATIC OFFSET ANALYSIS ===")
    
    # Calculate directional offsets
    known_matches_df['lat_offset'] = known_matches_df['latitude_google'] - known_matches_df['latitude_admin']
    known_matches_df['lon_offset'] = known_matches_df['longitude_google'] - known_matches_df['longitude_admin']

    print(f"Mean latitude offset:  {known_matches_df['lat_offset'].mean():.6f} degrees")
    print(f"Mean longitude offset: {known_matches_df['lon_offset'].mean():.6f} degrees")
    print(f"Std lat offset:  {known_matches_df['lat_offset'].std():.6f} degrees")
    print(f"Std lon offset: {known_matches_df['lon_offset'].std():.6f} degrees")
    
    # Check if offsets are consistent (suggest coordinate system issue)
    lat_consistency = known_matches_df['lat_offset'].std() / abs(known_matches_df['lat_offset'].mean()) if known_matches_df['lat_offset'].mean() != 0 else float('inf')
    lon_consistency = known_matches_df['lon_offset'].std() / abs(known_matches_df['lon_offset'].mean()) if known_matches_df['lon_offset'].mean() != 0 else float('inf')
    
    print(f"\nOffset Consistency (lower = more systematic):")
    print(f"Latitude consistency ratio: {lat_consistency:.2f}")
    print(f"Longitude consistency ratio: {lon_consistency:.2f}")
    
 # Calculate distances for plotting (this was missing)
    from geopy.distance import distance
    known_matches_df['distance_meters'] = known_matches_df.apply(
        lambda row: distance(
            (row['latitude_admin'], row['longitude_admin']),
            (row['latitude_google'], row['longitude_google'])
        ).meters, axis=1
    )

    # Plot offset patterns
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.scatter(known_matches_df['longitude_admin'], known_matches_df['latitude_admin'], 
                c=known_matches_df['distance_meters'], cmap='viridis', alpha=0.6)
    plt.colorbar(label='Distance (m)')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Discrepancy Patterns by Location')
    
    plt.subplot(1, 2, 2)
    plt.quiver(known_matches_df['longitude_admin'], known_matches_df['latitude_admin'],
               known_matches_df['lon_offset']*1000, known_matches_df['lat_offset']*1000,
               scale=1, scale_units='inches', alpha=0.6)
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Offset Directions (exaggerated)')
    
    plt.tight_layout()
    plt.show()
    
    return known_matches_df

In [ ]:
def check_common_coordinate_issues(website_df):
    """
    Check for swapped coordinates or different datum issues
    """
    print("\n=== COMMON COORDINATE ISSUES CHECK ===")
    
    # Check for swapped lat/lon (common error)
    potentially_swapped = website_df[
        (website_df['latitude'].abs() > 90) |  # Latitude should never be > 90
        (website_df['longitude'].abs() > 180)  # Longitude should never be > 180
    ]
    
    print(f"Potentially swapped coordinates: {len(potentially_swapped)}")
    
    # Check if coordinates might be in different format (DMS vs decimal)
    def looks_like_dms(coord_series):
        """Check if coordinates might be in Degrees Minutes Seconds format"""
        # DMS coordinates often have specific patterns
        return coord_series.apply(lambda x: 1000 < abs(x) < 100000).any()
    
    web_lat_dms = looks_like_dms(website_df['latitude'])
    web_lon_dms = looks_like_dms(website_df['longitude'])
    
    print(f"Website lat might be DMS format: {web_lat_dms}")
    print(f"Website lon might be DMS format: {web_lon_dms}")
    
    # Check for UTM or other projected coordinates
    utm_like = website_df[
        (website_df['latitude'] > 1000000) |  # UTM northing can be large
        (website_df['longitude'] > 1000000)   # UTM easting can be large
    ]
    print(f"Coordinates that look like UTM/projected: {len(utm_like)}")
    
    return potentially_swapped, utm_like

In [ ]:
#Step 4: Geographic Distribution Check
def check_geographic_distribution(website_df, google_df):
    """
    Verify both datasets cover similar geographic areas
    """
    print("\n=== GEOGRAPHIC DISTRIBUTION ===")
    
    # Calculate bounding boxes
    web_bbox = {
        'min_lat': website_df['latitude'].min(),
        'max_lat': website_df['latitude'].max(), 
        'min_lon': website_df['longitude'].min(),
        'max_lon': website_df['longitude'].max()
    }
    
    google_bbox = {
        'min_lat': google_df['latitude'].min(),
        'max_lat': google_df['latitude'].max(),
        'min_lon': google_df['longitude'].min(), 
        'max_lon': google_df['longitude'].max()
    }
    
    print("Website bounding box:")
    print(f"  Lat: {web_bbox['min_lat']:.4f} to {web_bbox['max_lat']:.4f}")
    print(f"  Lon: {web_bbox['min_lon']:.4f} to {web_bbox['max_lon']:.4f}")
    
    print("Google bounding box:")
    print(f"  Lat: {google_bbox['min_lat']:.4f} to {google_bbox['max_lat']:.4f}")
    print(f"  Lon: {google_bbox['min_lon']:.4f} to {google_bbox['max_lon']:.4f}")
    
    # Check if they're in completely different regions
    lat_overlap = min(web_bbox['max_lat'], google_bbox['max_lat']) - max(web_bbox['min_lat'], google_bbox['min_lat'])
    lon_overlap = min(web_bbox['max_lon'], google_bbox['max_lon']) - max(web_bbox['min_lon'], google_bbox['min_lon'])
    
    print(f"\nBounding box overlap:")
    print(f"  Latitude overlap: {lat_overlap:.4f} degrees")
    print(f"  Longitude overlap: {lon_overlap:.4f} degrees")
    
    if lat_overlap < 0 or lon_overlap < 0:
        print("🚨 WARNING: Datasets might be in completely different geographic regions!")
    
    return web_bbox, google_bbox

In [ ]:
# Run all coordinate system checks
print("🔍 RUNNING COORDINATE SYSTEM DIAGNOSTICS...")

# 1. Basic ranges
web_lat_issues, web_lon_issues = check_coordinate_systems(pids_geo_latlong, pids_geocoded)

# 2. Systematic offsets  
known_matches_df = check_systematic_offsets(known_matches)

# 3. Common issues
potentially_swapped, utm_like = check_common_coordinate_issues(pids_geo_latlong)

# 4. Geographic distribution
web_bbox, google_bbox = check_geographic_distribution(pids_geo_latlong, pids_geocoded)

print("\n" + "="*50)
print("SUMMARY OF FINDINGS:")
print("="*50)

if len(web_lat_issues) > 0 or len(web_lon_issues) > 0:
    print("❌ Coordinate range issues detected")
else:
    print("✅ Coordinates in expected WGS84 ranges")

if len(potentially_swapped) > 0:
    print("❌ Possible coordinate swapping detected") 
else:
    print("✅ No obvious coordinate swapping")

if len(utm_like) > 0:
    print("❌ Possible UTM/projected coordinates detected")
else:
    print("✅ No UTM/projected coordinates detected")

In [ ]:

def haversine_distance(coord1, coord2):
    """Calculate Haversine distance between two coordinates in meters"""
    return geopy.distance.distance(coord1, coord2).meters

def analyze_coordinate_discrepancies(known_matches_df):
    """
    Analyze the actual coordinate differences in your known matches
    """
    # Calculate distances for all known matches
    distances = []
    for idx, row in known_matches_df.iterrows():
        dist = haversine_distance(
            (row['latitude_admin'], row['longitude_admin']),
            (row['latitude_google'], row['longitude_google'])
        )
        distances.append(dist)
    
    distances = np.array(distances)
    
    # Analyze the distribution
    print("=== Coordinate Discrepancy Analysis ===")
    print(f"Mean distance: {np.mean(distances):.2f}m")
    print(f"Median distance: {np.median(distances):.2f}m")
    print(f"95th percentile: {np.percentile(distances, 95):.2f}m")
    print(f"99th percentile: {np.percentile(distances, 99):.2f}m")
    print(f"Max distance: {np.max(distances):.2f}m")
    
    # Plot distribution
    plt.figure(figsize=(10, 6))
    plt.hist(distances, bins=50, edgecolor='black', alpha=0.7)
    plt.axvline(np.percentile(distances, 95), color='red', linestyle='--', 
                label=f'95th percentile ({np.percentile(distances, 95):.1f}m)')
    plt.axvline(np.percentile(distances, 99), color='orange', linestyle='--', 
                label=f'99th percentile ({np.percentile(distances, 99):.1f}m)')
    plt.xlabel('Distance (meters)')
    plt.ylabel('Frequency')
    plt.xticks(range(0, int(np.max(distances)), 100))
    plt.title('Distribution of Coordinate Differences in Known Matches')
    plt.legend()
    plt.show()
    
    return distances

### cross checking the known matches as the coordinantes in the admin and google data for vary

In [ ]:
# Run the check
web_lat_issues, web_lon_issues = check_coordinate_systems(admin_unmatched, uncertain_pids)

In [ ]:
admin_unmatched.info()

### There are 1146 of the pools in  the admin dataset that are unmatched and there are 6072 in the google data unmatched. Using the already matched pids, we would analysis the difference in the coordinates and then create a radius in which we can consider the points as the same place


In [ ]:
#drop  rows in admin_unmatched where the coordinates are outside Germany
admin_unmatched = admin_unmatched[(admin_unmatched['latitude'] >= 47.270e0) & (admin_unmatched['latitude'] <= 55.058e0) & (admin_unmatched['longitude'] >= 5.866e0) & (admin_unmatched['longitude'] <= 15.041e0)]
admin_unmatched.info()

### drop the _Natürliche Badestelle_ pool_type from the admin_ unmatched

In [ ]:
admin_unmatched = admin_unmatched.drop(admin_unmatched[admin_unmatched['pool_type'] == 'Natürliche Badestelle'].index).reset_index(drop=True)
admin_unmatched.info()

### checking that  coordinates are in the same CRS for unmatched

In [ ]:
# Run the check
web_lat_issues, web_lon_issues = check_coordinate_systems(admin_unmatched, uncertain_pids)

In [ ]:
def check_systematic_offsets_between_datasets(admin_df, google_df, sample_size=1000):
    """
    Check for systematic offsets between two separate datasets by finding nearest neighbors
    
    Parameters:
    - admin_df: DataFrame with 'latitude', 'longitude' (1146 points)
    - google_df: DataFrame with 'latitude', 'longitude' (6072 points) 
    - sample_size: Number of points to sample for analysis (for performance)
    """
    print("\n=== SYSTEMATIC OFFSET ANALYSIS BETWEEN DATASETS ===")
    print(f"Admin unmatched points: {len(admin_df)}")
    print(f"Google uncertain points: {len(google_df)}")
    
    # Sample points for analysis (for performance with larger datasets)
    if len(admin_df) > sample_size:
        admin_sample = admin_df.sample(sample_size, random_state=42)
        print(f"Sampled {sample_size} admin points for analysis")
    else:
        admin_sample = admin_df
    
    if len(google_df) > sample_size:
        google_sample = google_df.sample(sample_size, random_state=42)
        print(f"Sampled {sample_size} google points for analysis")
    else:
        google_sample = google_df
    
    # Convert to radians for BallTree
    admin_rad = np.deg2rad(admin_sample[['latitude', 'longitude']].values)
    google_rad = np.deg2rad(google_sample[['latitude', 'longitude']].values)
    
    # Find nearest google point for each admin point
    tree = BallTree(google_rad, metric='haversine')
    distances, indices = tree.query(admin_rad, k=1)
    
    # Calculate offsets for nearest neighbors
    offsets = []
    for i, (admin_idx, (dist, google_idx)) in enumerate(zip(admin_sample.index, zip(distances, indices))):
        admin_lat = admin_sample.loc[admin_idx, 'latitude']
        admin_lon = admin_sample.loc[admin_idx, 'longitude']
        google_lat = google_sample.iloc[google_idx[0]]['latitude']
        google_lon = google_sample.iloc[google_idx[0]]['longitude']
        
        lat_offset = google_lat - admin_lat
        lon_offset = google_lon - admin_lon
        distance_m = dist[0] * 6371000  # Convert radians to meters
        
        offsets.append({
            'admin_lat': admin_lat,
            'admin_lon': admin_lon, 
            'google_lat': google_lat,
            'google_lon': google_lon,
            'lat_offset': lat_offset,
            'lon_offset': lon_offset,
            'distance_meters': distance_m
        })
    
    offsets_df = pd.DataFrame(offsets)
    
    # Calculate statistics
    print(f"\n--- Offset Statistics (based on {len(offsets_df)} nearest-neighbor pairs) ---")
    print(f"Mean latitude offset:  {offsets_df['lat_offset'].mean():.6f} degrees")
    print(f"Mean longitude offset: {offsets_df['lon_offset'].mean():.6f} degrees")
    print(f"Std lat offset:  {offsets_df['lat_offset'].std():.6f} degrees")
    print(f"Std lon offset: {offsets_df['lon_offset'].std():.6f} degrees")
    
    # Convert to meters for better interpretation
    deg_to_m_lat = 111000  # meters per degree latitude
    deg_to_m_lon = 71000   # meters per degree longitude (approx for Germany)
    
    print(f"\n--- Offsets in Meters ---")
    print(f"Mean latitude offset:  {offsets_df['lat_offset'].mean() * deg_to_m_lat:.2f} m")
    print(f"Mean longitude offset: {offsets_df['lon_offset'].mean() * deg_to_m_lon:.2f} m")
    print(f"Std lat offset:  {offsets_df['lat_offset'].std() * deg_to_m_lat:.2f} m")
    print(f"Std lon offset: {offsets_df['lon_offset'].std() * deg_to_m_lon:.2f} m")
    
    # Distance statistics
    print(f"\n--- Nearest Neighbor Distance Statistics ---")
    print(f"Mean distance: {offsets_df['distance_meters'].mean():.2f} m")
    print(f"Median distance: {offsets_df['distance_meters'].median():.2f} m")
    print(f"Min distance: {offsets_df['distance_meters'].min():.2f} m")
    print(f"Max distance: {offsets_df['distance_meters'].max():.2f} m")
    print(f"95th percentile: {offsets_df['distance_meters'].quantile(0.95):.2f} m")
    
    # Check if offsets are consistent (suggest coordinate system issue)
    lat_consistency = offsets_df['lat_offset'].std() / abs(offsets_df['lat_offset'].mean()) if offsets_df['lat_offset'].mean() != 0 else float('inf')
    lon_consistency = offsets_df['lon_offset'].std() / abs(offsets_df['lon_offset'].mean()) if offsets_df['lon_offset'].mean() != 0 else float('inf')
    
    print(f"\nOffset Consistency (lower = more systematic):")
    print(f"Latitude consistency ratio: {lat_consistency:.2f}")
    print(f"Longitude consistency ratio: {lon_consistency:.2f}")
    
    # Plotting
    plt.figure(figsize=(15, 5))
    
    # Plot 1: Distance distribution
    plt.subplot(1, 3, 1)
    plt.hist(offsets_df['distance_meters'], bins=50, edgecolor='black', alpha=0.7)
    plt.axvline(offsets_df['distance_meters'].median(), color='red', linestyle='--', 
                label=f'Median: {offsets_df["distance_meters"].median():.1f}m')
    plt.xlabel('Distance to Nearest Google Point (m)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Nearest Neighbor Distances')
    plt.legend()
    
    # Plot 2: Offset directions
    plt.subplot(1, 3, 2)
    # Sample for cleaner visualization
    plot_sample = offsets_df.sample(min(200, len(offsets_df)), random_state=42)
    plt.quiver(plot_sample['admin_lon'], plot_sample['admin_lat'],
               plot_sample['lon_offset']*10000, plot_sample['lat_offset']*10000,
               scale=1, scale_units='inches', alpha=0.6, color='blue')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Offset Directions (10000x exaggerated)')
    
    # Plot 3: Geographic distribution of distances
    plt.subplot(1, 3, 3)
    scatter = plt.scatter(offsets_df['admin_lon'], offsets_df['admin_lat'], 
                c=offsets_df['distance_meters'], cmap='viridis', alpha=0.6, s=10)
    plt.colorbar(scatter, label='Distance to Nearest Google Point (m)')
    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title('Geographic Distribution of Distances')
    
    plt.tight_layout()
    plt.show()
    
    # Interpretation
    print(f"\n=== INTERPRETATION ===")
    median_dist = offsets_df['distance_meters'].median()
    if median_dist < 50:
        print(f"✅ Good overlap: Median distance {median_dist:.1f}m suggests many potential matches")
    elif median_dist < 200:
        print(f"⚠️  Moderate overlap: Median distance {median_dist:.1f}m suggests some potential matches")
    else:
        print(f"❌ Poor overlap: Median distance {median_dist:.1f}m suggests few potential matches")
    
    return offsets_df

# Usage with your datasets:
offsets_analysis = check_systematic_offsets_between_datasets(admin_unmatched, uncertain_pids)

In [ ]:
admin_unmatched.groupby('pool_type').count()

In [ ]:
admin_unmatched[admin_unmatched['address'].isna()]#['pool_type'].unique()

### except for Bechthaler Weihe, the other 3 places are temporarily closed on google maps.
### Checking to see if the are in the old google maps pids using the unique identifier from the urls the freibad in  the dataset

In [ ]:
admin_nan_address_uid = ['0x47971f25d66878fb:0xa48b2416320b0ffe',  '0x479f17cdc3244de5:0xc377e7882b6f3aa0', '0x47babc74f7357927:0xa4d0d982bc815d23', '0x47a39157f5e7668f:0x8b1d5902ea9380f0']
merged_google_pids[merged_google_pids['unique_identifier'].isin(admin_nan_address_uid)]

In [ ]:
uncertain_pids[uncertain_pids['unique_identifier'].isin(admin_nan_address_uid)]

### Analysising the coordinate discrepancies in the  known matches to help with getting a radius for the unknown

In [ ]:
# Run this on your 5000 known matches
distances = analyze_coordinate_discrepancies(known_matches)

In [ ]:
def multi_tier_radius_matching(website_df, google_df, known_matches_df=None):
    """
    Smart multi-tier matching based on your actual data distribution
    """
    print("=== IMPLEMENTING MULTI-TIER MATCHING ===")
    
    # Use your known matches to set optimal radii
    if known_matches_df is not None:
        distances = known_matches_df['distance_meters']
        print(f"Based on {len(distances)} known matches:")
        print(f"25m radius captures: {len([d for d in distances if d <= 25])/len(distances)*100:.1f}%")
        print(f"50m radius captures: {len([d for d in distances if d <= 50])/len(distances)*100:.1f}%")
        print(f"100m radius captures: {len([d for d in distances if d <= 100])/len(distances)*100:.1f}%")
    
    # Conservative tiers based on your data
    tiers = [
        (25, 'high_confidence'),
        (50, 'medium_confidence'), 
        (100, 'low_confidence')
    ]
    
    all_matches = []
    remaining_website = website_df.copy()
    remaining_google = google_df.copy()
    
    for radius, confidence in tiers:
        print(f"\n--- Matching with {radius}m radius ({confidence}) ---")
        
        # Convert to radians for BallTree
        web_rad = np.deg2rad(remaining_website[['latitude', 'longitude']].values)
        google_rad = np.deg2rad(remaining_google[['latitude', 'longitude']].values)
        
        # Build BallTree for efficient radius queries
        tree = BallTree(google_rad, metric='haversine')
        
        # Find matches within radius
        indices = tree.query_radius(web_rad, r=radius/6371000)
        
        tier_matches = []
        matched_website_indices = set()
        matched_google_indices = set()
        
        for i, google_indices in enumerate(indices):
            if len(google_indices) > 0:
                # Take the closest match
                website_idx = remaining_website.index[i]
                closest_google_idx = google_indices[0]  # BallTree returns sorted by distance
                google_orig_idx = remaining_google.index[closest_google_idx]
                
                distance = haversine_distance(
                    (remaining_website.iloc[i]['latitude'], remaining_website.iloc[i]['longitude']),
                    (remaining_google.iloc[closest_google_idx]['latitude'], remaining_google.iloc[closest_google_idx]['longitude'])
                )
                
                tier_matches.append({
                    'website_index': website_idx,
                    'google_index': google_orig_idx,
                    'distance_meters': distance,
                    'confidence': confidence,
                    'latitude_admin': remaining_website.iloc[i]['latitude'],
                    'longitude_admin': remaining_website.iloc[i]['longitude'],
                    'latitude_google': remaining_google.iloc[closest_google_idx]['latitude'],
                    'longitude_google': remaining_google.iloc[closest_google_idx]['longitude']
                })
                
                matched_website_indices.add(website_idx)
                matched_google_indices.add(google_orig_idx)
        
        # Remove matched points from subsequent tiers
        remaining_website = remaining_website.drop(matched_website_indices)
        remaining_google = remaining_google.drop(matched_google_indices)
        
        print(f"Found {len(tier_matches)} matches")
        print(f"Remaining website points: {len(remaining_website)}")
        print(f"Remaining google points: {len(remaining_google)}")
        
        all_matches.extend(tier_matches)
    
    matches_df = pd.DataFrame(all_matches)
    
    print(f"\n=== FINAL RESULTS ===")
    print(f"Total matches found: {len(matches_df)}")
    print(f"Unmatched website points: {len(remaining_website)}")
    print(f"Unmatched google points: {len(remaining_google)}")
    
    # Breakdown by confidence level
    if len(matches_df) > 0:
        conf_counts = matches_df['confidence'].value_counts()
        for conf, count in conf_counts.items():
            print(f"{conf}: {count} matches")
    
    return matches_df, remaining_website, remaining_google

# Run the matching
matches_df, unmatched_website, unmatched_google = multi_tier_radius_matching(
    admin_pools,merged_google_pids, known_matches_df
)

In [ ]:
def correct_coordinates_in_place(df, lat_col='latitude', lon_col='longitude'):
    """
    Correct the coordinate columns in place by replacing first comma with dot and removing others
    """
    # Store admin_pools counts for reporting
    admin_pools_lat_problems = df[lat_col].astype(str).str.contains(',').sum()
    admin_pools_lon_problems = df[lon_col].astype(str).str.contains(',').sum()
    
    # Function to apply the transformation
    def transform_coordinate(value):
        if isinstance(value, str) and ',' in value:
            parts = value.split(',')
            return parts[0] + '.' + ''.join(parts[1:])
        return value
    
    # Apply transformations
    df[lat_col] = df[lat_col].apply(transform_coordinate)
    df[lon_col] = df[lon_col].apply(transform_coordinate)
    
    # Convert to float64
    df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
    
    # Report what was fixed
    print(f"Fixed {admin_pools_lat_problems} latitude values and {admin_pools_lon_problems} longitude values")
    print(f"Data types after correction:")
    print(f"  {lat_col}: {df[lat_col].dtype}")
    print(f"  {lon_col}: {df[lon_col].dtype}")
    
    return df

# Apply the correction to your admin_pools dataframe
print("Before correction:")
print(f"admin_pools latitude dtype: {admin_pools['latitude'].dtype}")
print(f"admin_pools longitude dtype: {admin_pools['longitude'].dtype}")

# Correct the dataframe IN PLACE
admin_pools = correct_coordinates_in_place(admin_pools)

print("\nAfter correction:")
print(f"Sample latitude values: {admin_pools['latitude'].head()}")
print(f"Sample longitude values: {admin_pools['longitude'].head()}")

In [ ]:
def find_true_unique_rows(original_df, pool_df):
    """
    Find rows in original that are not in pool_pids, accounting for coordinate duplicates
    """
    print("=== ANALYZING COORDINATE DUPLICATES ===")
    
    # Count unique coordinates in each dataset
    original_unique_coords = original_df[['latitude', 'longitude']].drop_duplicates()
    pool_unique_coords = pool_df[['latitude', 'longitude']].drop_duplicates()
    
    print(f"Original total rows: {len(original_df)}")
    print(f"Original unique coordinates: {len(original_unique_coords)}")
    print(f"Pool_pids total rows: {len(pool_df)}")
    print(f"Pool_pids unique coordinates: {len(pool_unique_coords)}")
    
    # Find coordinates in original but not in pool
    merged = original_unique_coords.merge(
        pool_unique_coords, 
        on=['latitude', 'longitude'], 
        how='left', 
        indicator=True
    )
    
    unique_coords = merged[merged['_merge'] == 'left_only']
    
    print(f"\nUnique coordinates in original (not in pool): {len(unique_coords)}")
    
    # Get the full rows from original that have these unique coordinates
    # We'll take just the first occurrence of each unique coordinate
    unique_rows = original_df.merge(
        unique_coords[['latitude', 'longitude']], 
        on=['latitude', 'longitude']
    ).drop_duplicates(subset=['latitude', 'longitude'])
    
    return unique_rows

true_unique_rows = find_true_unique_rows(admin_pools, merged_google_pids)
print(f"\nTrue unique rows (first occurrence of each unique coordinate):")
print(true_unique_rows)

In [ ]:

true_unique_rows.info()

In [ ]:
merged_all_pids[['latitude', 'longitude']]#.to_csv('merged_google_latlongs.csv', index=False)

In [ ]:
admin_pools[['latitude', 'longitude']]#.to_csv('admin_latlongs.csv', index=False)

In [ ]:
## pids collected forstate capitals only
reviews_gsc = pd.read_csv('/home/ethel/Documents/RESEARCH/swimming_pools_data_16gsc/annotations_12092025/all_swim_reviews_germany.csv')
reviews_gsc.info()

In [ ]:
pids_initial[pids_initial['place_id'].isin(pids_geocoded['place_id'])]

In [ ]:
notin_pids_geocoded = pids_geocoded[~pids_geocoded['place_id'].isin(pids_initial['place_id'])]
notin_pids_geocoded.info()

In [ ]:
# collected reviews
pids_initial_reviews = pd.read_parquet('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/public_swimmingpools_germany/clean_datasets/merged_reviews.parquet')
pids_initial_reviews.info()

In [ ]:
pids_initial[pids_initial['place_id'].isin(pids_initial_reviews['place_id'])][['place_id']].nunique()

In [ ]:
pids_geocoded[pids_geocoded['place_id'].isin(pids_initial_reviews['place_id'])][['place_id']].nunique()

In [ ]:
notin_pids_geocoded
#.dropna(subset=['place_rating'])

In [ ]:
notin_pids_geocoded[notin_pids_geocoded['place_rating'].isna()]

In [ ]:
notin_pids_geocoded[notin_pids_geocoded['place_rating']==0.0]

In [ ]:
'https://www.google.com/maps/place/Strandbad+Sandweier+Baden-Baden/@48.8193112,8.1760128,17z/data=!4m10!1m2!2m1!1sStrandbad+Sandweier!3m6!1s0x47971f25d66878fb:0xa48b2416320b0ffe!8m2!3d48.8174289!4d8.1807727!15sChNTdHJhbmRiYWQgU2FuZHdlaWVykgEYbGFrZV9zaG9yZV9zd2ltbWluZ19hcmVh4AEA!16s%2Fg%2F11mvsm83t5?authuser=0&entry=ttu&g_ep=EgoyMDI1MTEwOS4wIKXMDSoASAFQAw%3D%3D

In [ ]:
import pandas as pd
few_rewiews = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/848_pools/few_reviews.parquet")
few_rewiews.info()

In [ ]:
few_rewiews['place_id'].unique()

In [ ]:
few_rewiews_details = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/848_pools/more_reviews/clean_datasets/merged_details.parquet")
few_rewiews_details.info()

In [ ]:
few_rewiews_details['extracted_reviews']

In [ ]:
place_info=pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/reverse_pids/848_pools/general_poi_info.parquet")
place_info.info()